# NN eksperimenti — registry-based pipeline

Umjesto da svaki model ima svoju ćeliju sa cijelim training/eval kodom (kao u
`classic_models_cnn.ipynb`/`classic_models_cnn_fast.ipynb`), modeli se ovdje **registruju** u
`nn_utils.py` (`MODEL_REGISTRY`) i pozivaju po imenu preko `run_experiment(...)`.

Svaki poziv `run_experiment` sam radi train/val split, trening, work-level evaluaciju (soft-vote), i
**čuva sve** pod `output/{model_name}_{hash}/`:
- `config.json`, `metrics.json`, `history.json`
- `model.pt` (težine sa najboljim val_loss-om, ne zadnja epoha)
- `plots/loss_curve.png`, `plots/confusion_matrix.png`
- `report.pdf`

`hash` je izveden iz cijele konfiguracije (arhitektura + trening hiperparametri) — ista konfiguracija
uvijek ide u isti folder, drugačija dobija nov folder. Ništa se ne prepisuje slučajno.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

# Ucitavanje podataka

## Priprema metapodataka

Pregled izvorne tabele i raspodjele kompozitora nalazi se u [metadata.ipynb](metadata.ipynb).
`load_metadata()` iz `dataset.py` učitava CSV, formira `work_id` kao `composer + " | " + composition`
i zadržava kompozitore sa najmanje `MIN_WORKS_PER_COMPOSER` različitih djela (podrazumijevano 5, iz `config.py`).
Svi stavovi istog djela tako ostaju u istoj grupi tokom evaluacije.

Ova sveska se izvršava samostalno; nije potrebno prethodno pokretati `metadata.ipynb`.

In [ ]:
import numpy as np
import pandas as pd

from src.features.dataset import load_metadata
from src.utils.paths import AUDIO_DIR

metadata_filtered = load_metadata()

In [8]:
print(
    metadata_filtered.groupby("composer")["work_id"]
    .nunique()
    .sort_values()
)

composer
Brahms        8
Schubert      9
Mozart       11
Bach         30
Beethoven    55
Name: work_id, dtype: int64


---

## Mel-spektrogram dataset


In [ ]:
from src.features.spectrograms import make_spectrogram_dataset

X, y, groups = make_spectrogram_dataset(metadata_filtered, AUDIO_DIR)

print(X.shape)

---

## Pokretanje registrovanih modela

`cnn_small` i `crnn_small` su trenutno registrovani u `nn_utils.py` (`MODEL_REGISTRY`) — oba manji od
`SimpleCNN` korišćenog u ranijim notebook-ima, sa opcionim (Bi)GRU slojem nad vremenskom osom prije
poolinga. Svaki poziv niže je nezavisan eksperiment — parametri koji nisu navedeni uzimaju default iz
registry-ja.

In [ ]:
from src.model.nn_model import run_experiment

result_cnn = run_experiment(
    "cnn_small", X, y, groups,
    epochs=20, batch_size=64, dropout=0.6
)

In [11]:
result_crnn = run_experiment(
    "crnn_small", X, y, groups,
    epochs=20, batch_size=64, gru_hidden=32, dropout=0.6
)

epoch 1/20: 100%|██████████| 140/140 [00:58<00:00,  2.40it/s, acc=0.285, loss=1.55]


epoch 1/20: train_loss=1.5518 train_acc=0.2853 val_loss=1.6225 val_acc=0.1589


epoch 2/20: 100%|██████████| 140/140 [01:49<00:00,  1.28it/s, acc=0.41, loss=1.37] 


epoch 2/20: train_loss=1.3693 train_acc=0.4101 val_loss=1.6921 val_acc=0.2200


epoch 3/20: 100%|██████████| 140/140 [01:57<00:00,  1.20it/s, acc=0.47, loss=1.24] 


...

epoch 19/20: train_loss=0.9455 train_acc=0.6052 val_loss=2.3750 val_acc=0.1470


epoch 20/20: 100%|██████████| 140/140 [03:17<00:00,  1.41s/it, acc=0.617, loss=0.938]


epoch 20/20: train_loss=0.9384 train_acc=0.6174 val_loss=1.6708 val_acc=0.2797


predicting: 100%|██████████| 35/35 [00:17<00:00,  2.03it/s]
c:\Users\marko.vucic_ominimo\Desktop\music-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marko.vucic_ominimo\Desktop\music-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marko.vucic_ominimo\Desktop\music-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted sampl

saved to output\crnn_small_38be35b9  accuracy=0.3478  macro_f1=0.3354


---

## Poređenje

Rezultati su i printovani iznad i sačuvani na disku — ovo je samo brz pregled u samom notebook-u.

In [12]:
import pandas as pd

pd.DataFrame([
    {"model": "cnn_small", "accuracy": result_cnn["accuracy"], "macro_f1": result_cnn["macro_f1"], "run_dir": result_cnn["run_dir"]},
    {"model": "crnn_small", "accuracy": result_crnn["accuracy"], "macro_f1": result_crnn["macro_f1"], "run_dir": result_crnn["run_dir"]},
])

,model,accuracy,macro_f1,run_dir
0,cnn_small,0.478261,0.375000,output\cnn_small_38f41def
1,crnn_small,0.347826,0.335433,output\crnn_small_38be35b9


---

## RF + CNN ensemble

Kombinuje `cnn_small` (na spektrogramima) i Random Forest (na ručnim audio atributima iz
`feature_engineering.py`) — usrednjavanjem njihovih work-level vjerovatnoća. `run_ensemble_experiment`
sam obezbjeđuje da oba modela vide **identičan skup djela** u train/val split-u (`split_by_work`), iako
imaju različit broj segmenata po djelu (25s audio segmenti vs 10s spektrogram segmenti).

In [ ]:
from src.features.dataset import make_dataset

X_audio, y_audio, groups_audio = make_dataset(metadata_filtered, AUDIO_DIR)

print(X_audio.shape)

In [ ]:
from src.model.nn_model import run_ensemble_experiment

result_ensemble = run_ensemble_experiment(
    "cnn_small", X, y, groups, X_audio, y_audio, groups_audio,
    epochs=20, batch_size=64, dropout=0.6
)

---

## PANNs transfer learning


`make_panns_dataset` (audio se ponovo dekodira na 32kHz — PANNs očekuje tu stopu, ne 44.1kHz kao ostatak
projekta, pa se ne može iskoristiti postojeći keš) računa 2048-dim embedding po segmentu preko pretreniranog
Cnn14 modela (treniran na AudioSet-u, milioni zvučnih isječaka). `run_panns_experiment` onda trenira samo
mali RF klasifikator na vrhu tih (zamrznutih) embedinga — to je "transfer learning" dio: ne treniramo
enkoder od nule, samo iskorišćavamo već naučenu reprezentaciju.

In [ ]:
import time
from src.features.panns_features import make_panns_dataset

# small subset first - confirms the checkpoint download works and gives a real per-recording timing
# before committing to the full 302-recording dataset
_sample = metadata_filtered.head(10)

_start = time.time()
_Xs, _ys, _gs = make_panns_dataset(_sample, AUDIO_DIR, use_cache=False)
_elapsed = time.time() - _start

print(f"{len(_sample)} recordings -> {_Xs.shape} in {_elapsed:.1f}s ({_elapsed/len(_sample):.1f}s/recording)")
print(f"estimated full dataset (302 recordings): ~{_elapsed/len(_sample)*302/60:.1f} min")

In [16]:
X_panns, y_panns, groups_panns = make_panns_dataset(metadata_filtered, AUDIO_DIR)

print(X_panns.shape)

Processing PANNs embeddings: 100%|██████████| 302/302 [2:39:00<00:00, 31.59s/it]  


(11096, 2048)


In [ ]:
from src.model.nn_model import run_panns_experiment

result_panns = run_panns_experiment(X_panns, y_panns, groups_panns)

---

## Puno poređenje

In [18]:
rows = []
for name, r in [("cnn_small", result_cnn), ("crnn_small", result_crnn),
                ("ensemble_rf_cnn", result_ensemble), ("panns_rf", result_panns)]:
    if r.get("status") == "ok":
        rows.append({"model": name, "accuracy": r["accuracy"], "macro_f1": r["macro_f1"], "run_dir": r["run_dir"]})
    else:
        rows.append({"model": name, "accuracy": None, "macro_f1": None, "run_dir": r["run_dir"], "status": r.get("status")})

pd.DataFrame(rows)

,model,accuracy,macro_f1,run_dir
0,cnn_small,0.478261,0.375000,output\cnn_small_38f41def
1,crnn_small,0.347826,0.335433,output\crnn_small_38be35b9
2,ensemble_rf_cnn,0.590909,0.415152,output\ensemble_rf_cnn_small_f8157c20
3,panns_rf,0.500000,0.305263,output\panns_rf_0ceb0cb9
